In [0]:
from pyspark.sql.functions import col, to_date

# Define parameters via widgets
dbutils.widgets.text("catalog", "dbr_dev", "1. Catalog Name")
dbutils.widgets.text("bronze_schema", "valeriimatviiv_bronze", "2. Bronze Schema")
dbutils.widgets.text("silver_schema", "valeriimatviiv_silver", "3. Silver Schema")

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")

source_table = f"{catalog}.{bronze_schema}.nasdaq_price_bronze"
target_table = f"{catalog}.{silver_schema}.nasdaq_price_silver"

# 1. Read Bronze Price Data
df_bronze = spark.read.table(source_table)

# 2. Apply Type Casting, Rename Columns, and Handle Nulls
df_silver = (
    df_bronze
    .withColumn("TradeDate", to_date(col("Date"), "yyyy-MM-dd"))
    .withColumn("Open", col("Open").cast("double"))
    .withColumn("High", col("High").cast("double"))
    .withColumn("Low", col("Low").cast("double"))
    .withColumn("Close", col("Close").cast("double"))
    .withColumn("Volume", col("Volume").cast("long"))
    .select(
        "Symbol",
        "TradeDate",
        "Open",
        "High",
        "Low",
        "Close",
        "Volume",
        "_source_file",
        "_ingest_timestamp"
    )
    .dropna(subset=["Symbol", "TradeDate", "Close"])
    # Deduplicate by primary key (Symbol, TradeDate)
    .dropDuplicates(["Symbol", "TradeDate"])
    .orderBy("Symbol", "TradeDate")
)

# 3. Write Cleaned Data to Silver Delta Table
df_silver.write.format("delta").mode("overwrite").saveAsTable(target_table)

print(f"Successfully processed Silver Price table: '{target_table}'")

In [0]:
# catalog = dbutils.widgets.get("catalog")
# silver_schema = dbutils.widgets.get("silver_schema")
# target_table = f"{catalog}.{silver_schema}.nasdaq_price_silver"

# df_silver = spark.read.table(target_table)

# print(f"--- Silver Price Table Record Count: {df_silver.count()} ---")
# print("--- Schema Breakdown ---")
# df_silver.printSchema()

# print("--- Preview Data ---")
# display(df_silver.limit(10))